# Loss ablation -- both phases, 5-fold cross-validation

K-fold sibling of `both_phase_ablation.ipynb`, which screens three loss arms
(`dice_ce`, `tversky_ce`, `dice_ce_boundary`) at `phase='both'` on one fixed
34/8 validation split. That notebook's paired comparisons are pinned to n=8
validation patients (or n=7 x SEEDS with seed-averaging), which caps the
per-chamber Wilcoxon tests below Holm significance no matter how consistent
an effect is (exact floor at n=8 is 2/2\*\*8 = 0.0078; the x12 Holm penalty
per class puts every per-chamber test out of reach). This notebook exists to
see whether per-chamber trends -- especially LA/RA (the `tversky_ce` target)
and HD95/ASSD (the `dice_ce_boundary` target) -- survive once the paired
sample grows from 8 patients to the full ~42-patient train+val pool.

Mechanics, not a new comparison: same phase (`'both'`), same three arms, same
frozen `SPLIT_SEED=0`, same `AUGMENT=True` (every arm trains with
augmentation on, matching `both_phase_ablation.ipynb` -- the loss axis is
isolated from the augmentation axis, which `run_both_phase_kfold.sh` already
covers separately). The only thing that changes is how the non-test pool is
partitioned -- 5 disjoint folds instead of one fixed 34/8 split -- via
`train.split_patients_kfold()` (see `train.py`). The held-out test set is
IDENTICAL to every existing `both_*` run at this split seed; k-fold never
touches it, and the split logic guarantees this (same patient list, same
seed, same shuffle, same test carve-out -- only the remaining pool is
partitioned differently).

**Training does not happen in this kernel.** Folds (and arms) are independent
models with nothing to synchronize between them, so they parallelize across
GPUs as separate OS processes -- `train.py`'s GPU selection is via
`CUDA_VISIBLE_DEVICES`, fixed once a process starts, so a single notebook
kernel is pinned to one GPU for its lifetime. Launch
`run_both_phase_loss_ablation_kfold.sh` from a terminal to actually train;
this notebook confirms the split, then reads results back from disk once
training has finished -- same resume-from-disk design as
`both_phase_augmentation_kfold_diagnostics.ipynb`.

**Full tier-2/tier-3 diagnostic toolkit**, carried over from that notebook:
a cross-fold consistency check, precision/recall decomposition,
size-stratified performance, a slice-position performance profile,
predictive-entropy uncertainty (summary numbers AND pixel-wise maps), and
terminal-slice failure inspection -- on top of the pooled macro/per-chamber
comparison against the `dice_ce` baseline. Nothing here needs retraining or
touches the test set; it all reads already-trained fold `best.pth` files.
Every arm-specific section is generalized from 2 arms to `len(ARMS)`, so
adding a fourth loss (e.g. `focal_tversky_ce`) later costs one line in
`ARMS`, nothing else.

## Setup

In [ ]:
import os

# only needed for the final test-set scoring cell at the bottom -- the split
# confirmation and pooled-analysis cells above it never touch the GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
print(f"Targeting GPU: {os.environ['CUDA_VISIBLE_DEVICES']}")


In [ ]:
import sys
sys.path.append('.')

import json
import logging
import numpy as np
import torch
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
import train
from unet import UNet
from utils import metrics
from utils.data_loading import VolumeMRIDataset
from utils.losses import LOSS_REGISTRY

train.dir_img = Path('./data/imgs/')
train.dir_mask = Path('./data/masks/')
train.dir_checkpoint = Path('./checkpoints/')

print('available arms:', sorted(LOSS_REGISTRY))


## Experiment configuration

`ARMS`, `PHASE` and `SPLIT_SEED` are fixed to match `both_phase_ablation.ipynb`
exactly, so both notebooks' results sit in the same comparison space and share
the same frozen test set. `K_FOLDS=5` is the only new knob -- same choice as
`both_phase_augmentation_kfold_diagnostics.ipynb` (the pool has ~42 patients;
5 folds keeps each fold's own val set from getting too small for its own
checkpoint-selection signal, while staying cheap enough to run).

Arms, same rationale as `loss_ablation.ipynb` / `both_phase_ablation.ipynb`:

- `dice_ce` (**baseline**) -- corrected Dice + CE at 1:1. Everything else is
  measured against this.
- `tversky_ce` -- alpha=0.3, beta=0.7, so a false negative costs more than a
  false positive. The standard lever for under-segmented thin-walled
  structures; LA/RA/EAT are the likely beneficiaries.
- `dice_ce_boundary` -- Kervadec boundary loss, region weight ramping
  1.0 -> 0.01. The only arm that targets HD95/ASSD directly rather than
  overlap; its CPU-side distance transforms make it ~3x the wall-clock cost
  of the other two (5 foreground classes now, not 4 -- see
  `both_phase_ablation.ipynb`).

In [ ]:
PHASE      = 'both'
ARMS       = ['dice_ce', 'tversky_ce', 'dice_ce_boundary']
BASELINE_ARM = 'dice_ce'
K_FOLDS    = 5

SPLIT_SEED   = 0        # FROZEN. Identical to loss_ablation.ipynb, both_phase_ablation.ipynb
                         # and both_phase_augmentation_kfold_diagnostics.ipynb -- do not change;
                         # this is what keeps the test set identical across every both_* notebook.
EPOCHS       = 40
BATCH_SIZE   = 8
LR           = 1e-5
IMG_SCALE    = 1.0
N_CLASSES    = train.PHASE_N_CLASSES[PHASE]     # 6 for 'both' (bg + LV/RV/LA/RA/EAT)
N_CHANNELS   = train.PHASE_N_CHANNELS[PHASE]    # 2 for 'both' (water, fat)
AMP          = True
AUGMENT      = True     # every arm trains identically-augmented -- see both_phase_ablation.ipynb
SELECT_ON    = 'macro_dice'
LR_SCHEDULE  = 'poly'
TEST_PERCENT = 0.15

RUN_PREFIX = 'abl'

def run_name_for(arm, fold):
    return f'{RUN_PREFIX}_{arm}_kfold{fold}'

# one closure per arm, fold -> run_name -- this is the shape every shared-setup
# function below (predict_patient, pool_slice_table, ...) expects, matching
# both_phase_augmentation_kfold_diagnostics.ipynb's baseline_run_name/aug_run_name
RUN_NAME_FN = {arm: (lambda f, arm=arm: run_name_for(arm, f)) for arm in ARMS}

# train_model prefixes non-water run names with '{phase}_' internally -- see
# CLAUDE.md ("water keeps unprefixed run names"). Mirrors run_dir_for() in
# both_phase_augmentation_kfold_diagnostics.ipynb.
def run_dir_for(run_name):
    full_name = run_name if PHASE == 'water' else f'{PHASE}_{run_name}'
    return train.dir_checkpoint / full_name

# fixed categorical order, slots 1-3 of the project's default palette (validated
# CVD-safe across all pairs for exactly these three slots -- see the dataviz
# skill's palette.md). Never cycled: dice_ce is always blue, tversky_ce always
# orange, dice_ce_boundary always aqua, in every figure below.
ARM_COLORS = {'dice_ce': '#2a78d6', 'tversky_ce': '#eb6834', 'dice_ce_boundary': '#1baf7a'}
assert set(ARMS) <= set(LOSS_REGISTRY), set(ARMS) - set(LOSS_REGISTRY)
assert BASELINE_ARM in ARMS

print(f'{K_FOLDS}-fold, {EPOCHS} epochs each, {len(ARMS)} arms '
      f'({len(ARMS) * K_FOLDS} runs total):')
for arm in ARMS:
    for f in range(K_FOLDS):
        print(f'  {run_dir_for(run_name_for(arm, f)).name:<28} (loss={arm})')

# rough wall-clock, from loss_ablation.ipynb's measured ~68 min/40-epoch run
# for dice_ce/tversky_ce and ~3x that for dice_ce_boundary's CPU-side distance
# transforms, x K_FOLDS
est_h = sum((68 * 3 if a == 'dice_ce_boundary' else 68) for a in ARMS) * K_FOLDS / 60
print(f'\n~{est_h:.0f} h total serial compute (divide by however many GPUs run_both_phase_loss_ablation_kfold.sh gets)')

# every diagnostic plot below saves here automatically, in addition to plt.show() --
# so nothing needs a live kernel/GPU to be VIEWED again later, only to be regenerated
POOLED_DIR = train.dir_checkpoint / f'{PHASE}_loss_ablation_kfold_pooled'
PLOTS_DIR = POOLED_DIR / 'plots'
SLICES_DIR = PLOTS_DIR / 'slices'
SLICES_DIR.mkdir(parents=True, exist_ok=True)   # also creates POOLED_DIR/PLOTS_DIR


## Dataset

Built once, same as the seed-based notebook. `VolumeMRIDataset(..., phase='both')`
shares its on-disk cache (`data/preprocessed_cache/`, `_both`-suffixed files)
with `both_phase_augmentation.ipynb` and `both_phase_ablation.ipynb` -- already
fully populated by those notebooks' runs, so this is a cache-read, not a fresh
DICOM/NIfTI parse.

In [ ]:
dataset = VolumeMRIDataset(train.dir_img, train.dir_mask, scale=IMG_SCALE, phase=PHASE)
print(f'{len(dataset.mask_file_for)} patients, {len(dataset.index)} slices')
print(f'mask values: {dataset.mask_values}')


### Confirm the folds and the test set before spending any GPU time

Two separate things get checked here: that the 5 folds are disjoint and cover
the whole non-test pool (basic correctness of `split_patients_kfold`), and
that the test set it produces -- the one scored exactly once, at the very
end -- is byte-for-byte the same 8 patients every other `both_*` notebook at
`split_seed=0` already uses. If the second assertion ever fails, something
about the dataset or the split logic changed and the frozen test set can no
longer be trusted; do not proceed past that without finding out why.

In [ ]:
EXPECTED_TEST_PATIENTS = ['CADRE_1113', 'CADRE_1116_first', 'CADRE_1523', 'CADRE_1532',
                          'CADRE_1743', 'CADRE_1744', 'CADRE_1859', 'CADRE_1867']

all_val_patients = []
test_patients = None
for f in range(K_FOLDS):
    train_idx, val_idx, test_idx = train.split_patients_kfold(
        dataset, fold=f, k_folds=K_FOLDS, test_percent=TEST_PERCENT, seed=SPLIT_SEED
    )
    fold_test = sorted({dataset.index[i][0] for i in test_idx})
    fold_val = sorted({dataset.index[i][0] for i in val_idx})
    fold_train = sorted({dataset.index[i][0] for i in train_idx})

    if test_patients is None:
        test_patients = fold_test
    assert fold_test == test_patients, f'fold {f} has a different test set!'
    assert not (set(fold_val) & set(fold_train)), f'fold {f}: train/val overlap'
    assert not (set(fold_val) & set(fold_test)), f'fold {f}: val/test overlap'

    all_val_patients.append(set(fold_val))
    print(f'fold {f}: train {len(fold_train):>2} / val {len(fold_val):>2}   val = {fold_val}')

assert test_patients == EXPECTED_TEST_PATIENTS, \
    f'test set does not match the frozen split! got {test_patients}'

pool = set(dataset.mask_file_for) - set(test_patients)
union = set().union(*all_val_patients)
assert union == pool, 'folds do not cover the whole train+val pool'
for i in range(K_FOLDS):
    for j in range(i + 1, K_FOLDS):
        assert not (all_val_patients[i] & all_val_patients[j]), f'folds {i} and {j} overlap'

print(f'\nOK: test set matches the frozen split ({len(test_patients)} patients), '
      f'{K_FOLDS} folds are disjoint and cover the {len(pool)}-patient pool')


## Shared setup: predicting any pool patient

Used by both Tier 2 and Tier 3 below, so it's defined once, early, rather than
buried inside the diagnostics section. Same principle as
`both_phase_augmentation_kfold_diagnostics.ipynb`: every pool patient has
exactly one fold whose model never trained on them -- predict that patient
with THAT fold's model, nothing else, regardless of which arm's `run_name_fn`
is passed in. Also computes per-pixel predictive entropy alongside the
prediction -- same forward pass, one extra softmax + reduction, no
retraining, no extra model calls.

In [ ]:
from collections import defaultdict

# which fold holds each pool patient out, once -- this never changes for a
# given SPLIT_SEED/K_FOLDS, independent of which arm you're looking at
_FOLD_OF = {}
for f in range(K_FOLDS):
    _, val_idx, _ = train.split_patients_kfold(dataset, fold=f, k_folds=K_FOLDS,
                                                test_percent=TEST_PERCENT, seed=SPLIT_SEED)
    for p in {dataset.index[i][0] for i in val_idx}:
        _FOLD_OF[p] = f

_PATIENT_IDXS = defaultdict(list)
for i, (p, _) in enumerate(dataset.index):
    _PATIENT_IDXS[p].append(i)

_model_cache = {}   # run_name -> loaded model, so re-visualizing doesn't reload weights
_pred_cache = {}    # (run_name, patient_id) -> (pred_vol, gt_vol, entropy_vol)


def fold_ready(run_name_fn, fold):
    """Whether this fold's best.pth exists yet -- lets pool-scanning functions
    skip unfinished folds instead of crashing on the first missing checkpoint."""
    return (run_dir_for(run_name_fn(fold)) / 'best.pth').exists()


def load_fold_model(run_name_fn, fold):
    """Load (and cache, for this kernel) one fold's best.pth for one arm."""
    run_name = run_name_fn(fold)
    if run_name not in _model_cache:
        model = UNet(n_channels=N_CHANNELS, n_classes=N_CLASSES, bilinear=False)
        model = model.to(memory_format=torch.channels_last).to(device=device)
        state_dict = torch.load(run_dir_for(run_name) / 'best.pth', map_location=device)
        state_dict.pop('mask_values', None)      # injected by train.py, not a real weight
        model.load_state_dict(state_dict)
        model.eval()
        _model_cache[run_name] = model
    return _model_cache[run_name]


def predict_patient(patient_id, run_name_fn):
    """Predict one patient's whole volume -- prediction, ground truth, AND
    per-pixel predictive entropy -- with the ONE fold model that never trained
    on them. Raises if patient_id isn't in the train+val pool (e.g. it's a
    held-out test patient -- deliberately not reachable from here)."""
    if patient_id not in _FOLD_OF:
        raise ValueError(f'{patient_id} is not a train+val pool patient '
                         '(it may be in the held-out test set)')
    fold = _FOLD_OF[patient_id]
    run_name = run_name_fn(fold)
    key = (run_name, patient_id)
    if key not in _pred_cache:
        model = load_fold_model(run_name_fn, fold)
        pred_vol, gt_vol, entropy_vol = metrics.predict_volume(
            model, dataset, _PATIENT_IDXS[patient_id], device, amp=AMP,
            batch_size=BATCH_SIZE, return_entropy=True)
        _pred_cache[key] = (pred_vol, gt_vol, entropy_vol)
    pred_vol, gt_vol, entropy_vol = _pred_cache[key]
    return pred_vol, gt_vol, entropy_vol, fold


def slice_scores(pred_vol, gt_vol):
    """Mean foreground Dice per slice; nan for slices with no GT foreground
    (an empty slice trivially scores 1.0 otherwise and would swamp 'best')."""
    scores = []
    for s in range(pred_vol.shape[0]):
        cs = [metrics.dice_binary(pred_vol[s] == c, gt_vol[s] == c) for c in range(1, N_CLASSES)]
        cs = [x for x in cs if not np.isnan(x)]
        scores.append(float(np.mean(cs)) if cs else float('nan'))
    return np.array(scores)


def pool_slice_table(run_name_fn):
    """One row per (patient, slice) across the whole pool: Dice, mean predictive
    entropy over the GT foreground, and normalized position (0 = first slice of
    that patient's volume, 1 = last -- normalized because volumes have different
    slice counts, so raw index isn't comparable across patients). Predictions are
    cached, so re-calling this for an arm already visualized elsewhere is free.
    Patients whose fold hasn't finished training yet are skipped, not an error --
    same partial-progress tolerance as load_fold_rows() below."""
    rows, skipped = [], 0
    for patient_id in sorted(_FOLD_OF):
        if not fold_ready(run_name_fn, _FOLD_OF[patient_id]):
            skipped += 1
            continue
        pred_vol, gt_vol, entropy_vol, fold = predict_patient(patient_id, run_name_fn)
        n_slices = pred_vol.shape[0]
        scores = slice_scores(pred_vol, gt_vol)
        for s in range(n_slices):
            if not np.isfinite(scores[s]):
                continue
            fg = gt_vol[s] > 0
            mean_entropy = float(entropy_vol[s][fg].mean()) if fg.any() else float('nan')
            rows.append({'patient_id': patient_id, 'slice_idx': s, 'fold': fold,
                        'position': s / max(1, n_slices - 1),
                        'dice': float(scores[s]), 'mean_entropy': mean_entropy})
    if skipped:
        print(f'pool_slice_table({run_name_fn(0).rsplit("_kfold", 1)[0]}): '
              f'{skipped}/{len(_FOLD_OF)} patients skipped -- their fold has not finished training yet')
    return rows


def pool_entropy_by_class(run_name_fn):
    """Mean predictive entropy within each PREDICTED structure, per patient/class
    -- same (patient, class) shape as the dice/precision/recall rows already in
    `pooled_rows`, so it can be joined against them directly. Same skip-if-not-
    finished tolerance as pool_slice_table()."""
    rows, skipped = [], 0
    for patient_id in sorted(_FOLD_OF):
        if not fold_ready(run_name_fn, _FOLD_OF[patient_id]):
            skipped += 1
            continue
        pred_vol, gt_vol, entropy_vol, fold = predict_patient(patient_id, run_name_fn)
        for cls in range(1, N_CLASSES):
            mask = pred_vol == cls
            mean_entropy = float(entropy_vol[mask].mean()) if mask.any() else float('nan')
            rows.append({'patient_id': patient_id, 'cls': cls,
                        'class_name': train.PHASE_CLASS_NAMES[PHASE][cls],
                        'mean_entropy': mean_entropy})
    if skipped:
        print(f'pool_entropy_by_class({run_name_fn(0).rsplit("_kfold", 1)[0]}): '
              f'{skipped}/{len(_FOLD_OF)} patients skipped -- their fold has not finished training yet')
    return rows


print(f'{len(_FOLD_OF)} pool patients indexed to their held-out fold; '
      f'models/predictions load lazily and are cached per kernel session')


## Training

Not run from this kernel -- see the intro cell for why. From a terminal on a
node with GPUs free:

```bash
bash run_both_phase_loss_ablation_kfold.sh          # uses every GPU nvidia-smi reports
bash run_both_phase_loss_ablation_kfold.sh 0 2 3    # or pin it to specific GPU indices
```

This launches all `len(ARMS) * K_FOLDS` (15) runs as separate `train.py`
processes, distributed round-robin across whatever GPU indices you give it.
Each run is independent and resumable -- rerunning the script skips any run
whose `run_config.json` already has `'finished'`, same spirit as `run_arm()`'s
`already_done()` in `both_phase_ablation.ipynb`. Check progress with
`nvidia-smi` / `wandb`; come back to this notebook once the runs you care
about are done -- everything below reads purely from disk, so a fresh kernel
is fine.

## Results

In [ ]:
def load_fold_rows(run_name_fn):
    """Concatenate val_metrics_per_patient.csv across every FINISHED fold.

    Unlike averaging across seeds (loss_ablation.ipynb's arm_rows), folds are
    DISJOINT patients -- pooling them is a plain concatenation, not an average.
    Each patient appears in exactly one fold's val set, so this produces one
    row per (patient, class) covering the whole train+val pool, same shape as
    a single run's per-patient CSV but with ~42 patients instead of ~7-8.
    """
    rows, folds_found = [], []
    for f in range(K_FOLDS):
        run_dir = run_dir_for(run_name_fn(f))
        csv_path = run_dir / 'val_metrics_per_patient.csv'
        cfg_path = run_dir / 'run_config.json'
        if not (csv_path.exists() and cfg_path.exists()):
            continue
        if 'finished' not in json.loads(cfg_path.read_text()):
            continue
        rows.extend(metrics.load_per_patient(csv_path))
        folds_found.append(f)
    return rows, folds_found


pooled_rows = {}
for arm in ARMS:
    rows, folds_found = load_fold_rows(RUN_NAME_FN[arm])
    pooled_rows[arm] = rows
    patients = sorted({r['patient_id'] for r in rows})
    status = folds_found if len(folds_found) == K_FOLDS else f'{folds_found} (incomplete)'
    print(f'{arm:<18} folds finished: {status}   {len(patients)} patients pooled')


### Per-class, side by side

Same layout as `both_phase_ablation.ipynb`, but `mean +/- SD` is now over the
pooled train+val patients (up to ~42) instead of the 7-8 validation patients
-- only arms with every fold finished are shown.

In [ ]:
ORDER = ['LV', 'RV', 'LA', 'RA', 'EAT', 'all_classes_macro']

summaries = {}
for arm in ARMS:
    _, folds_found = load_fold_rows(RUN_NAME_FN[arm])
    if len(folds_found) == K_FOLDS and pooled_rows[arm]:
        summaries[arm] = {r['class_name']: r for r in metrics.summarise(pooled_rows[arm])}

have = [a for a in ARMS if a in summaries]
if len(have) < len(ARMS):
    print('Not every arm has all K_FOLDS finished yet -- only complete arms are summarised.\n')

n_pool = len(set(dataset.mask_file_for) - set(test_patients))
for metric, unit, better in [('dice', '', 'higher'),
                             ('hd95_mm', ' mm', 'lower'),
                             ('assd_mm', ' mm', 'lower')]:
    print(f'\n=== {metric}{unit}  ({better} is better)   '
          f'mean +/- SD over up to {n_pool} pooled train+val patients ===')
    print(f'{"":<20}' + ''.join(f'{a:>24}' for a in have))
    for cls in ORDER:
        cells_ = []
        for a in have:
            r = summaries[a].get(cls)
            if r is None:
                cells_.append(f'{"n/a":>24}')
                continue
            cells_.append(f'{r[f"{metric}_mean"]:>13.4f} +/- {r[f"{metric}_sd"]:<7.4f}')
        print(f'{cls:<20}' + ''.join(cells_))


print(f'\n=== n_missed  (ground truth present, prediction empty -- lower is better) ===')
print(f'{"":<20}' + ''.join(f'{a:>24}' for a in have))
for cls in ORDER:
    cells_ = []
    for a in have:
        r = summaries[a].get(cls)
        cells_.append(f'{"n/a":>24}' if r is None else f'{r["n_missed"]:>24}')
    print(f'{cls:<20}' + ''.join(cells_))


## Figures

In [ ]:
ORDER_FIG = ['LV', 'RV', 'LA', 'RA', 'EAT']
METRIC = 'dice'   # 'dice' | 'precision' | 'recall' | 'hd95_mm' | 'assd_mm'

import matplotlib.pyplot as plt

arms_present = [a for a in ARMS if pooled_rows.get(a)]

if not arms_present:
    print('No arm has any finished folds yet -- nothing to plot.')
else:
  jitter = np.random.default_rng(0)
  fig, axes = plt.subplots(1, len(ORDER_FIG), figsize=(4.3 * len(ORDER_FIG), 4.8), sharey=True)
  axes = np.atleast_1d(axes)
  for ax, cls in zip(axes, ORDER_FIG):
    data = [[r[METRIC] for r in pooled_rows[a]
             if r['class_name'] == cls and np.isfinite(r[METRIC])]
            for a in arms_present]

    box = ax.boxplot(data, tick_labels=arms_present, showmeans=True, widths=0.6, patch_artist=True)
    for patch, a in zip(box['boxes'], arms_present):
        patch.set_facecolor(ARM_COLORS[a])
        patch.set_alpha(0.35)
    for i, (vals, a) in enumerate(zip(data, arms_present), start=1):
        ax.scatter(jitter.normal(i, 0.045, len(vals)), vals, color=ARM_COLORS[a], alpha=0.7, s=18)
    ax.set_title(cls)
    ax.tick_params(axis='x', rotation=90)
    ax.set_ylabel(METRIC if ax is axes[0] else '')
  fig.suptitle(f'{METRIC} by class, pooled across {K_FOLDS} folds ({len(arms_present)} arms)')
  fig.tight_layout()
  save_path = PLOTS_DIR / f'boxplot_{METRIC}.png'
  fig.savefig(save_path, dpi=150, bbox_inches='tight')
  plt.show()
  print(f'saved to {save_path}')


## Paired comparison: each arm vs the baseline, pooled across folds

`compare_runs` pairs on `(patient_id, class_name)`, same test as
`both_phase_ablation.ipynb` uses -- the difference is the shared pool is now
every train+val patient (~42), not just the 7-8 validation patients, because
each patient's per-arm scores come from that patient's OWN held-out fold's
model (only the loss differs between arms at a given fold), so the pairing is
still valid patient-for-patient. One `compare_runs` call per non-baseline arm
against `dice_ce`, each with its own internal Holm correction across its own
12-test per-chamber family -- same as the pairwise loop in
`loss_ablation.ipynb` / `both_phase_ablation.ipynb`, not one combined family
across arms.

**Caveat, carried over from `both_phase_augmentation_kfold_diagnostics.ipynb`:**
patients held out together in one fold are scored by that fold's one trained
model each, so they share that model's training-noise -- the pooled n is not
fully independent in the textbook sense (see Bengio & Grandvalet 2004 on the
lack of an unbiased k-fold CV variance estimator). Read the p-values below as
more informative than the 7-8-patient version, not as exact. The macro row is
still the single pre-specified primary endpoint per arm; per-chamber rows are
descriptive.

In [ ]:
# POOLED_DIR was already created in the config cell above
pooled_paths = {}
for arm in ARMS:
    path = POOLED_DIR / f'{arm}_val_metrics_per_patient.csv'
    metrics.write_csv(pooled_rows[arm], path)
    pooled_paths[arm] = path
print('pooled CSVs written to', POOLED_DIR)

baseline_patients = {r['patient_id'] for r in pooled_rows.get(BASELINE_ARM, [])}
if not baseline_patients:
    print(f'\n{BASELINE_ARM} has no finished folds yet -- nothing to compare against.')
else:
    for arm in ARMS:
        if arm == BASELINE_ARM:
            continue
        if not pooled_rows.get(arm):
            print(f'\n== {arm}: not run yet, skipping')
            continue

        arm_patients = {r['patient_id'] for r in pooled_rows[arm]}
        if arm_patients != baseline_patients:
            print(f'\nWARNING: {BASELINE_ARM} has {len(baseline_patients)} patients pooled, '
                  f'{arm} has {len(arm_patients)} -- compare_runs will only use the '
                  f'{len(baseline_patients & arm_patients)} patients shared by both. Wait for all '
                  f'folds to finish on both arms before trusting this comparison.')

        print(f'\n===== {BASELINE_ARM}  vs  {arm}  (pooled train+val) =====')
        comparison = metrics.compare_runs(pooled_paths[BASELINE_ARM], pooled_paths[arm],
                                          label_a=BASELINE_ARM, label_b=arm)
        print(metrics.format_comparison(comparison, BASELINE_ARM, arm))


### Cross-fold consistency (still Tier 1)

The pooled comparison above treats every patient as one data point, but it's
worth also checking the more basic question directly: does EVERY fold agree
on which arm is better, or is the pooled win driven by one or two folds while
others disagree? Each fold's own `best_val_macro_dice` (already recorded in
its `run_config.json`, no extra computation) answers this -- one panel per
non-baseline arm, one line per fold connecting that fold's `dice_ce` score to
its own score for that arm.

In [ ]:
import matplotlib.pyplot as plt

fold_scores = {arm: [] for arm in ARMS}
for arm in ARMS:
    for f in range(K_FOLDS):
        cfg_path = run_dir_for(RUN_NAME_FN[arm](f)) / 'run_config.json'
        score = float('nan')
        if cfg_path.exists():
            cfg = json.loads(cfg_path.read_text())
            score = cfg.get('best_val_macro_dice', float('nan'))
        fold_scores[arm].append(score)

other_arms = [a for a in ARMS if a != BASELINE_ARM]
fig, axes = plt.subplots(1, len(other_arms), figsize=(5 * len(other_arms), 4.5), sharey=True)
axes = np.atleast_1d(axes)

any_plotted = False
for ax, arm in zip(axes, other_arms):
    folds_plotted = 0
    for f in range(K_FOLDS):
        b, a = fold_scores[BASELINE_ARM][f], fold_scores[arm][f]
        if not (np.isfinite(b) and np.isfinite(a)):
            continue
        ax.plot([0, 1], [b, a], color='#8a8a86', alpha=0.7, linewidth=1.6, zorder=1)
        ax.scatter([0, 1], [b, a], color=[ARM_COLORS[BASELINE_ARM], ARM_COLORS[arm]], s=60, zorder=2)
        ax.annotate(f'fold {f}', (1.03, a), va='center', fontsize=9, color='#52514e')
        folds_plotted += 1
    ax.set_xlim(-0.15, 1.35)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([BASELINE_ARM, arm])
    ax.set_title(f'{folds_plotted}/{K_FOLDS} folds finished')
    if folds_plotted:
        any_plotted = True
        n_agree = sum(1 for f in range(K_FOLDS)
                     if np.isfinite(fold_scores[BASELINE_ARM][f]) and np.isfinite(fold_scores[arm][f])
                     and fold_scores[arm][f] > fold_scores[BASELINE_ARM][f])
        print(f'{arm} scored higher than {BASELINE_ARM} in {n_agree}/{folds_plotted} finished folds')

axes[0].set_ylabel('best val macro Dice')
if any_plotted:
    fig.suptitle('Per-fold macro Dice: baseline -> each arm')
    fig.tight_layout()
    save_path = PLOTS_DIR / 'cross_fold_consistency.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'saved to {save_path}')
else:
    print('No fold has finished on both an arm and the baseline yet -- nothing to plot.')


## Tier 2 -- explanatory diagnostics

Stratified, quantitative views that explain *why* the Tier-1 numbers look the
way they do, rather than just confirming that they do. None of this decides
"which arm wins" on its own -- it's context for the decision above.

### Precision / recall decomposition

Dice is precision and recall's harmonic mean, so it can't tell you whether an
arm is fixing over-segmentation or under-segmentation. This is also the only
direct readout of whether `tversky_ce`'s beta>alpha did what it was designed
to do (raise recall). Reads the same pooled per-patient rows already loaded
in the Results section above.

In [ ]:
print(f'{"class":<8}' + ''.join(f'{a + " precision":>20}{a + " recall":>20}' for a in ARMS))
for cls in ORDER_FIG:
    row_cells = []
    for a in ARMS:
        rows = [r for r in pooled_rows.get(a, []) if r['class_name'] == cls]
        p = np.nanmean([r['precision'] for r in rows]) if rows else float('nan')
        r_ = np.nanmean([r['recall'] for r in rows]) if rows else float('nan')
        row_cells.append(f'{p:>20.4f}{r_:>20.4f}')
    print(f'{cls:<8}' + ''.join(row_cells))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
x = np.arange(len(ORDER_FIG))
width = 0.8 / len(ARMS)
for ax, metric, title in zip(axes, ['precision', 'recall'], ['Precision', 'Recall']):
    for i, a in enumerate(ARMS):
        vals = []
        for cls in ORDER_FIG:
            rows = [r for r in pooled_rows.get(a, []) if r['class_name'] == cls]
            vals.append(np.nanmean([r[metric] for r in rows]) if rows else np.nan)
        offset = (i - (len(ARMS) - 1) / 2) * width
        ax.bar(x + offset, vals, width, label=a, color=ARM_COLORS[a])
    ax.set_xticks(x)
    ax.set_xticklabels(ORDER_FIG)
    ax.set_title(title)
    ax.set_ylim(0, 1)
axes[0].set_ylabel('mean across pool')
axes[0].legend(frameon=False)
fig.suptitle('Precision / recall by class, pooled across folds')
fig.tight_layout()
save_path = PLOTS_DIR / 'precision_recall.png'
fig.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'saved to {save_path}')


### Size-stratified performance

Tests whether an arm's gain concentrates on small/hard structures -- the
natural question for `tversky_ce` (recall lever) and `dice_ce_boundary`
(distance-based) alike. Pairs each (patient, class) between an arm and the
baseline -- same pairing logic `compare_runs` uses internally -- and
correlates ground-truth structure size (mL, from `gt_voxels` * `voxel_ml`,
already columns in every per-patient CSV) against the Dice difference. One
panel per non-baseline arm.

In [ ]:
from scipy.stats import spearmanr

def size_stratified_gain(rows_a, rows_b, metric='dice'):
    a = {(r['patient_id'], r['class_name']): r for r in rows_a}
    b = {(r['patient_id'], r['class_name']): r for r in rows_b}
    shared = sorted(set(a) & set(b))
    sizes, gains, classes = [], [], []
    for k in shared:
        ra, rb = a[k], b[k]
        if not (np.isfinite(ra[metric]) and np.isfinite(rb[metric])):
            continue
        sizes.append(ra['gt_voxels'] * ra['voxel_ml'])
        gains.append(rb[metric] - ra[metric])
        classes.append(k[1])
    return np.array(sizes), np.array(gains), classes


# same class-color mapping as show_slice / the auto-saved best/worst-slice
# PNGs every run already produces -- kept consistent across the notebook
class_colors = {'LV': 'tab:red', 'RV': 'tab:green', 'LA': 'tab:blue',
                'RA': 'tab:orange', 'EAT': 'tab:purple'}

other_arms = [a for a in ARMS if a != BASELINE_ARM]
fig, axes = plt.subplots(1, len(other_arms), figsize=(6.5 * len(other_arms), 5), sharey=True)
axes = np.atleast_1d(axes)

any_plotted = False
for ax, arm in zip(axes, other_arms):
    sizes, gains, classes = size_stratified_gain(pooled_rows.get(BASELINE_ARM, []), pooled_rows.get(arm, []))
    if len(sizes) <= 2:
        ax.set_title(f'{arm}: not enough shared pairs yet')
        continue
    any_plotted = True
    rho, p = spearmanr(sizes, gains)
    print(f'{arm} vs {BASELINE_ARM} -- Spearman(size, Dice gain): rho={rho:.3f}, p={p:.4f} '
          f'(n={len(sizes)} patient x class pairs)')
    for cls in ORDER_FIG:
        mask = np.array(classes) == cls
        ax.scatter(sizes[mask], gains[mask], color=class_colors[cls], label=cls, s=28, alpha=0.75)
    ax.axhline(0, color='#52514e', linewidth=1, linestyle='--')
    ax.set_xlabel('ground-truth structure size (mL)')
    ax.set_title(f'{arm} vs {BASELINE_ARM}  (rho={rho:.2f}, p={p:.3f})')

axes[0].set_ylabel('Dice gain (arm - baseline)')
if any_plotted:
    axes[-1].legend(frameon=False, title='class')
    fig.suptitle('Size-stratified performance change')
    fig.tight_layout()
    save_path = PLOTS_DIR / 'size_stratified_gain.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'saved to {save_path}')
    print('Negative rho -> smaller structures gain more from that arm relative to the baseline.')
else:
    print('Not enough shared (patient, class) pairs yet -- wait for more folds to finish.')


### Slice-position performance profile

Whether performance systematically drops at the terminal (first/last) slices
of a volume, or holds steady through the middle. `position` is normalized per
patient (0 = first slice, 1 = last) by `pool_slice_table` above, so patients
with different slice counts are still comparable. All arms overlaid on one
figure, since the shapes of the curves relative to each other are the point.

In [ ]:
def position_profile(run_name_fn, n_bins=10):
    table = pool_slice_table(run_name_fn)
    if not table:
        return None
    positions = np.array([r['position'] for r in table])
    dices = np.array([r['dice'] for r in table])
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_idx = np.clip(np.digitize(positions, bin_edges) - 1, 0, n_bins - 1)
    means = [dices[bin_idx == b].mean() if (bin_idx == b).any() else np.nan for b in range(n_bins)]
    counts = [int((bin_idx == b).sum()) for b in range(n_bins)]
    centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    return centers, means, counts


fig, ax = plt.subplots(figsize=(8, 5))
any_plotted = False
for arm in ARMS:
    profile = position_profile(RUN_NAME_FN[arm])
    if profile is None:
        print(f'{arm}: no fold has finished yet, skipped')
        continue
    any_plotted = True
    centers, means, counts = profile
    ax.plot(centers, means, marker='o', color=ARM_COLORS[arm], linewidth=2, markersize=6, label=arm)

if any_plotted:
    ax.set_xlabel('normalized slice position (0 = first slice, 1 = last)')
    ax.set_ylabel('mean Dice (foreground classes)')
    ax.set_title(f'Dice vs. slice position, pooled across the CV pool')
    ax.legend(frameon=False)
    fig.tight_layout()
    save_path = PLOTS_DIR / 'slice_position_profile.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'saved to {save_path}')
else:
    print('No arm has any finished folds yet -- nothing to plot.')


### Uncertainty summary (per patient/class mean entropy)

A scalar per (patient, class) -- mean predictive entropy within the region the
model actually predicted as that class -- so it can sit in the same tables as
dice/precision/recall rather than only existing as a picture. Useful as a
quick check of whether high uncertainty tracks low Dice (expected) or whether
some arm is confidently wrong (a worse sign than being unsure). One block per
arm.

In [ ]:
for arm in ARMS:
    entropy_rows = pool_entropy_by_class(RUN_NAME_FN[arm])
    if not entropy_rows:
        print(f'{arm}: no fold has finished yet, skipped\n')
        continue
    dice_by_key = {(r['patient_id'], r['class_name']): r['dice'] for r in pooled_rows.get(arm, [])}

    print(f'=== {arm} ===')
    print(f'{"class":<8}{"mean entropy":>16}{"n":>6}')
    by_class = defaultdict(list)
    for r in entropy_rows:
        if np.isfinite(r['mean_entropy']):
            by_class[r['class_name']].append(r['mean_entropy'])
    for cls in ORDER_FIG:
        vals = by_class.get(cls, [])
        if vals:
            print(f'{cls:<8}{np.mean(vals):>16.4f}{len(vals):>6}')

    paired = [(dice_by_key[(r['patient_id'], r['class_name'])], r['mean_entropy'])
             for r in entropy_rows
             if (r['patient_id'], r['class_name']) in dice_by_key and np.isfinite(r['mean_entropy'])]
    if len(paired) > 2:
        dices_, entropies_ = zip(*paired)
        rho, p = spearmanr(dices_, entropies_)
        print(f'Spearman correlation, Dice vs mean entropy: rho={rho:.3f}, p={p:.4f}  (n={len(paired)})')
    print()

print('Expected: negative rho for every arm (lower Dice where the model is also less confident). '
     'A near-zero or positive rho would mean that arm is confidently wrong somewhere.')


## Diagnostics (Tier 3): visualize predictions across the pool

Qualitative/visual checks -- the setup (model loading, prediction, entropy,
caching) already happened in the shared setup section near the top of this
notebook; everything below just uses it.

### Global worst/best slices, pooled across the whole CV set

Not per-fold -- across all ~42 patients at once, each scored by the model that
legitimately held them out. `ARM_TO_INSPECT` picks which arm's own predictions
rank the slices; the drill-in below then shows every arm side by side on
whichever slices come out worst for it.

In [ ]:
def pool_slice_ranking(run_name_fn):
    """Worst-to-best slices across the pool, built from pool_slice_table
    (predictions are cached, so this is free if the table was already computed
    for this arm elsewhere in the notebook). Returns (score, patient_id,
    slice_idx, fold) tuples."""
    table = pool_slice_table(run_name_fn)
    entries = [(r['dice'], r['patient_id'], r['slice_idx'], r['fold']) for r in table]
    entries.sort(key=lambda e: e[0])
    return entries


ARM_TO_INSPECT = 'tversky_ce'   # <- swap for any arm in ARMS

ranking = pool_slice_ranking(RUN_NAME_FN[ARM_TO_INSPECT])
if not ranking:
    print(f'No fold has finished for {ARM_TO_INSPECT} yet.')
else:
    print(f'worst 6 slices, pooled across the CV set ({ARM_TO_INSPECT}):')
    for score, pid, s, fold in ranking[:6]:
        print(f'  {score:.3f}  {pid:<20} slice {s:<4} (fold {fold})')

    print(f'\nbest 6 slices:')
    for score, pid, s, fold in ranking[-6:][::-1]:
        print(f'  {score:.3f}  {pid:<20} slice {s:<4} (fold {fold})')


### Drill in: ground truth vs. one or more arms, side by side

`show_slice` takes a dict of `{label: run_name_fn}` -- pass one arm to just
look at its predictions, or several (e.g. all of `RUN_NAME_FN`) to see exactly
how the loss choice changed a specific case. Every panel uses the SAME fold's
model for a given patient (fold membership doesn't depend on the arm), so any
visible difference is attributable to the loss, not to a different train/val
split. Entropy panels roughly double the figure width per arm shown, so the
worst-slice sweep below defaults to all arms without entropy, and a smaller
baseline-vs-`ARM_TO_INSPECT` pair with entropy for the closer look.

In [ ]:
def show_slice(patient_id, slice_idx, arms, title_extra='', show_entropy=False):
    """Plot ground truth + one prediction panel per arm for one slice, and
    optionally an entropy heatmap panel per arm. `arms` is {label: run_name_fn}.
    """
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Patch

    colors = ['none', 'tab:red', 'tab:green', 'tab:blue', 'tab:orange', 'tab:purple']
    cmap = ListedColormap(colors[:N_CLASSES])
    class_names = train.PHASE_CLASS_NAMES[PHASE]

    preds, entropies, gt, fold = {}, {}, None, None
    for label, run_name_fn in arms.items():
        pred_vol, gt_vol, entropy_vol, f = predict_patient(patient_id, run_name_fn)
        preds[label] = pred_vol[slice_idx]
        entropies[label] = entropy_vol[slice_idx]
        gt, fold = gt_vol[slice_idx], f

    ordered = sorted(_PATIENT_IDXS[patient_id], key=lambda i: dataset.index[i][1])
    img = dataset[ordered[slice_idx]]['image'].numpy()[0]

    n_panels = 1 + len(arms) * (2 if show_entropy else 1)
    fig, axes = plt.subplots(1, n_panels, figsize=(4.2 * n_panels, 4.5))
    axes = np.atleast_1d(axes)

    axes[0].imshow(img, cmap='gray')
    axes[0].imshow(np.ma.masked_equal(gt, 0), cmap=cmap, vmin=0, vmax=N_CLASSES - 1, alpha=0.5)
    axes[0].set_title('ground truth')
    axes[0].axis('off')

    col = 1
    for label, pred in preds.items():
        ax = axes[col]
        ax.imshow(img, cmap='gray')
        ax.imshow(np.ma.masked_equal(pred, 0), cmap=cmap, vmin=0, vmax=N_CLASSES - 1, alpha=0.5)
        dice_here = np.nanmean([metrics.dice_binary(pred == c, gt == c) for c in range(1, N_CLASSES)])
        ax.set_title(f'{label} (Dice={dice_here:.3f})')
        ax.axis('off')
        col += 1

    if show_entropy:
        for label, ent in entropies.items():
            ax = axes[col]
            ax.imshow(img, cmap='gray')
            # sequential single-hue colormap for magnitude (uncertainty), not a rainbow
            im = ax.imshow(ent, cmap='viridis', alpha=0.6, vmin=0)
            ax.set_title(f'{label} entropy')
            ax.axis('off')
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            col += 1

    handles = [Patch(color=colors[c], label=class_names.get(c, f'class_{c}')) for c in range(1, N_CLASSES)]
    fig.legend(handles=handles, loc='lower center', ncol=len(handles), frameon=False)
    fig.suptitle(f'{patient_id} slice {slice_idx} (fold {fold}){title_extra}')
    fig.tight_layout(rect=(0, 0.08, 1, 1))

    tag = '-'.join(arms) + ('_entropy' if show_entropy else '')
    save_path = SLICES_DIR / f'{patient_id}_slice{slice_idx}_{tag}.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'saved to {save_path}')
    return save_path


# example: worst few slices found above, EVERY arm side by side, no entropy
ALL_ARMS = {arm: RUN_NAME_FN[arm] for arm in ARMS}
if ranking:
    for score, pid, s, fold in ranking[:3]:
        show_slice(pid, s, ALL_ARMS,
                  title_extra=f'  [{ARM_TO_INSPECT} worst-slice pick, Dice={score:.3f}]')


### Terminal-slice failure inspection

Filters the same per-slice table to just the first/last 15% of each patient's
volume, to check specifically whether -- and why -- performance degrades at
the terminal slices (small partial-volume structures near the apex, ambiguous
valve-plane anatomy near the base, etc.). Drill-in here uses just baseline vs
`ARM_TO_INSPECT` with entropy maps, to keep each figure to 5 panels rather
than `1 + 2*len(ARMS)`.

In [ ]:
EDGE_BAND = 0.15   # slices within this fraction of either end count as "terminal"

table = pool_slice_table(RUN_NAME_FN[ARM_TO_INSPECT])
if not table:
    print(f'No fold has finished for {ARM_TO_INSPECT} yet.')
else:
    terminal = [r for r in table if r['position'] <= EDGE_BAND or r['position'] >= 1 - EDGE_BAND]
    interior = [r for r in table if EDGE_BAND < r['position'] < 1 - EDGE_BAND]

    print(f'terminal slices (position <= {EDGE_BAND} or >= {1 - EDGE_BAND}): '
          f'{len(terminal)} slices, mean Dice = {np.mean([r["dice"] for r in terminal]):.4f}')
    print(f'interior slices:                                          '
          f'{len(interior)} slices, mean Dice = {np.mean([r["dice"] for r in interior]):.4f}')

    worst_terminal = sorted(terminal, key=lambda r: r['dice'])[:3]
    print(f'\nworst 3 terminal slices:')
    for r in worst_terminal:
        print(f'  {r["dice"]:.3f}  {r["patient_id"]:<20} slice {r["slice_idx"]:<4} '
             f'(position {r["position"]:.2f}, fold {r["fold"]})')
    compare_pair = {BASELINE_ARM: RUN_NAME_FN[BASELINE_ARM], ARM_TO_INSPECT: RUN_NAME_FN[ARM_TO_INSPECT]}
    for r in worst_terminal:
        show_slice(r['patient_id'], r['slice_idx'], compare_pair,
                  title_extra=f'  [terminal slice, position={r["position"]:.2f}]', show_entropy=True)


### EAT-focused slice inspection: water + fat phase, GT, all arms

EAT is consistently the worst-performing structure across every arm (dice
~0.75-0.77 vs 0.92+ for the four chambers) and, unlike the per-chamber effects
above, it doesn't move much with the loss choice either -- suggesting the
bottleneck may be more about the structure itself (diffuse, low-contrast,
and defined mainly by the fat-phase channel rather than the water phase the
other four chambers live in) than something an under/over-segmentation-tuned
loss can fix.

Ranks slices by **EAT's own Dice**, not the mean-over-classes Dice
`pool_slice_ranking` above uses -- a slice can have fine overall Dice while
EAT specifically is bad, which is exactly the case this cell is built to
surface. For each worst slice it shows both input channels (`water_img`,
`fat_img`, channel 0/1 of `dataset[...]['image']` -- EAT is defined mostly by
the fat phase, so seeing both side by side is the point) plus ground truth
and every arm's prediction, with each arm's title reporting its own EAT Dice
for that slice specifically. Same fold-consistency guarantee as `show_slice`
above: every panel for a given patient uses that patient's one held-out
fold's model, so differences between arms are attributable to the loss only.

In [ ]:
EAT_CLS = 5   # BOTH_CLASS_NAMES[5] == 'EAT' -- see train.py
N_EAT_SLICES = 8   # how many worst-EAT-Dice slices to pull up
EAT_RANK_ARM = BASELINE_ARM   # <- which arm's EAT Dice picks the worst slices; arms are still shown side by side


def pool_class_slice_ranking(run_name_fn, cls):
    """Worst-to-best slices for ONE class, pooled across the CV set -- unlike
    pool_slice_ranking (mean Dice over every foreground class), this ranks by
    cls's own Dice, so a slice with fine overall Dice but bad EAT specifically
    still surfaces. Same nan-if-absent convention as slice_scores(): a slice
    with no cls in its GT is excluded rather than trivially scoring 1.0."""
    entries = []
    for patient_id in sorted(_FOLD_OF):
        if not fold_ready(run_name_fn, _FOLD_OF[patient_id]):
            continue
        pred_vol, gt_vol, _, fold = predict_patient(patient_id, run_name_fn)
        for s in range(pred_vol.shape[0]):
            gt_slice = gt_vol[s]
            if not (gt_slice == cls).any():
                continue
            d = metrics.dice_binary(pred_vol[s] == cls, gt_slice == cls)
            if np.isfinite(d):
                entries.append((d, patient_id, s, fold))
    entries.sort(key=lambda e: e[0])
    return entries


def show_eat_slice(patient_id, slice_idx, arms, eat_cls=EAT_CLS, title_extra=''):
    """Water phase, fat phase, ground truth, and every arm's prediction for one
    slice -- built for EAT specifically, so (unlike show_slice) it shows BOTH
    input channels rather than just channel 0, and reports each arm's EAT-only
    Dice rather than the mean-over-classes Dice in its panel title. Overlays
    the full class colormap (not just EAT) so confusion with the neighboring
    chambers -- RA in particular, given EAT's anatomical position -- stays
    visible too."""
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Patch

    colors = ['none', 'tab:red', 'tab:green', 'tab:blue', 'tab:orange', 'tab:purple']
    cmap = ListedColormap(colors[:N_CLASSES])
    class_names = train.PHASE_CLASS_NAMES[PHASE]

    preds, gt, fold = {}, None, None
    for label, run_name_fn in arms.items():
        pred_vol, gt_vol, _, f = predict_patient(patient_id, run_name_fn)
        preds[label] = pred_vol[slice_idx]
        gt, fold = gt_vol[slice_idx], f

    ordered = sorted(_PATIENT_IDXS[patient_id], key=lambda i: dataset.index[i][1])
    img = dataset[ordered[slice_idx]]['image'].numpy()   # (2, H, W): water, fat
    water_img, fat_img = img[0], img[1]

    n_panels = 2 + 1 + len(arms)   # water, fat, GT, one per arm
    fig, axes = plt.subplots(1, n_panels, figsize=(4.0 * n_panels, 4.5))
    axes = np.atleast_1d(axes)

    axes[0].imshow(water_img, cmap='gray')
    axes[0].set_title('water phase')
    axes[0].axis('off')

    axes[1].imshow(fat_img, cmap='gray')
    axes[1].set_title('fat phase')
    axes[1].axis('off')

    axes[2].imshow(fat_img, cmap='gray')
    axes[2].imshow(np.ma.masked_equal(gt, 0), cmap=cmap, vmin=0, vmax=N_CLASSES - 1, alpha=0.5)
    axes[2].set_title('ground truth (on fat phase)')
    axes[2].axis('off')

    col = 3
    for label, pred in preds.items():
        ax = axes[col]
        ax.imshow(fat_img, cmap='gray')
        ax.imshow(np.ma.masked_equal(pred, 0), cmap=cmap, vmin=0, vmax=N_CLASSES - 1, alpha=0.5)
        eat_dice = metrics.dice_binary(pred == eat_cls, gt == eat_cls)
        ax.set_title(f'{label} (EAT Dice={eat_dice:.3f})')
        ax.axis('off')
        col += 1

    handles = [Patch(color=colors[c], label=class_names.get(c, f'class_{c}')) for c in range(1, N_CLASSES)]
    fig.legend(handles=handles, loc='lower center', ncol=len(handles), frameon=False)
    fig.suptitle(f'{patient_id} slice {slice_idx} (fold {fold}){title_extra}')
    fig.tight_layout(rect=(0, 0.08, 1, 1))

    tag = '-'.join(arms) + '_eat'
    save_path = SLICES_DIR / f'{patient_id}_slice{slice_idx}_{tag}.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'saved to {save_path}')
    return save_path


eat_ranking = pool_class_slice_ranking(RUN_NAME_FN[EAT_RANK_ARM], EAT_CLS)
if not eat_ranking:
    print(f'No fold has finished for {EAT_RANK_ARM} yet.')
else:
    print(f'worst {N_EAT_SLICES} EAT slices, pooled across the CV set (ranked by {EAT_RANK_ARM} EAT Dice):')
    for score, pid, s, fold in eat_ranking[:N_EAT_SLICES]:
        print(f'  {score:.3f}  {pid:<20} slice {s:<4} (fold {fold})')

    for score, pid, s, fold in eat_ranking[:N_EAT_SLICES]:
        show_eat_slice(pid, s, ALL_ARMS,
                       title_extra=f'  [{EAT_RANK_ARM} worst-EAT-slice pick, EAT Dice={score:.3f}]')


### The single worst EAT slice, in its slice-neighborhood context

Takes just the #1 worst EAT slice from the ranking above and looks at its
immediate neighbors (`+/- EAT_NEIGHBOR_WINDOW` slices in the same volume) --
the question this is built to answer is specifically: **is EAT actually
visible/coherent in the slices right next to the failure, even though the
model (which only ever sees one slice at a time) got the center slice
wrong?** If so, that's direct visual evidence that the missing information
was sitting one slice away the whole time -- exactly what a 2.5D model (fed
a small stack of adjacent slices instead of one) would have access to and a
plain 2D model structurally cannot.

Every arm is shown at every neighboring slice, not just the ranking arm --
if all three arms fail similarly at the center slice despite differing
losses (which the per-chamber comparisons above already suggest for EAT),
that reinforces that this is a missing-context problem, not something a
different loss function could have fixed. The failure column is outlined in
red in every row so it stays identifiable at a glance.

In [ ]:
EAT_NEIGHBOR_WINDOW = 2   # slices on each side of the worst slice to include
NEIGHBORHOOD_ARM = EAT_RANK_ARM   # <- which arm's ranking picks the "worst" slices; all arms still shown in the grid
N_EAT_NEIGHBORHOOD_SWEEP = 6   # how many worst EAT slices to walk through, not just #1 -- a single
                               # case is anecdotal, a repeated pattern across several is evidence


def show_eat_slice_neighborhood(patient_id, center_slice, arms, eat_cls=EAT_CLS,
                                 window=EAT_NEIGHBOR_WINDOW, title_extra=''):
    """Fat phase, ground truth, and every arm's prediction across a small window
    of consecutive slices centered on one EAT failure. Built to check whether
    EAT is visible/coherent in the neighboring slices even where the model --
    which only ever sees one slice at a time -- missed it at the center. If GT
    shows EAT clearly a slice or two away from a slice where every arm fails,
    that's direct visual evidence the missing information was one slice away
    the whole time, which is exactly what a 2.5D model (fed a small stack of
    adjacent slices) has access to and a 2D model structurally cannot.
    Predictions are read out of the cached whole-volume predict_patient() call
    per arm, so this costs no extra forward passes beyond what's already run.
    """
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Patch

    colors = ['none', 'tab:red', 'tab:green', 'tab:blue', 'tab:orange', 'tab:purple']
    cmap = ListedColormap(colors[:N_CLASSES])
    class_names = train.PHASE_CLASS_NAMES[PHASE]

    preds, gts, fold, n_slices = {}, {}, None, None
    for label, run_name_fn in arms.items():
        pred_vol, gt_vol, _, f = predict_patient(patient_id, run_name_fn)
        preds[label], gts[label], fold = pred_vol, gt_vol, f
        n_slices = pred_vol.shape[0]

    slice_range = [s for s in range(center_slice - window, center_slice + window + 1)
                   if 0 <= s < n_slices]
    ordered = sorted(_PATIENT_IDXS[patient_id], key=lambda i: dataset.index[i][1])
    fat_imgs = {s: dataset[ordered[s]]['image'].numpy()[1] for s in slice_range}

    first_arm = next(iter(arms))
    row_labels = ['fat phase', 'ground truth'] + list(arms)
    n_rows, n_cols = len(row_labels), len(slice_range)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.0 * n_cols, 3.0 * n_rows), squeeze=False)

    for col, s in enumerate(slice_range):
        gt_slice = gts[first_arm][s]   # ground truth is identical across arms
        panels = [('fat phase', None), ('ground truth', gt_slice)] + \
                 [(label, preds[label][s]) for label in arms]
        for row, (row_label, overlay) in enumerate(panels):
            ax = axes[row, col]
            ax.imshow(fat_imgs[s], cmap='gray')
            if overlay is not None:
                ax.imshow(np.ma.masked_equal(overlay, 0), cmap=cmap, vmin=0, vmax=N_CLASSES - 1, alpha=0.5)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(s == center_slice)
                if s == center_slice:
                    spine.set_edgecolor('tab:red')
                    spine.set_linewidth(2.5)
            if col == 0:
                ax.set_ylabel(row_label, fontsize=10)
            if row == 0:
                title = f'slice {s}' + ('  (FAILURE)' if s == center_slice else '')
                ax.set_title(title, fontweight='bold' if s == center_slice else 'normal',
                              color='tab:red' if s == center_slice else 'black')
            elif row == 1:
                has_eat = (gt_slice == eat_cls).any()
                ax.set_title('EAT present' if has_eat else 'no EAT in GT', fontsize=9,
                              color='black' if has_eat else '#8a8a86')
            else:
                eat_dice = metrics.dice_binary(overlay == eat_cls, gt_slice == eat_cls)
                ax.set_title(f'EAT Dice={eat_dice:.3f}', fontsize=9)

    handles = [Patch(color=colors[c], label=class_names.get(c, f'class_{c}')) for c in range(1, N_CLASSES)]
    fig.legend(handles=handles, loc='lower center', ncol=len(handles), frameon=False)
    fig.suptitle(f'{patient_id}  slices {slice_range[0]}-{slice_range[-1]} (fold {fold}){title_extra}')
    fig.tight_layout(rect=(0, 0.04, 1, 0.97))

    save_path = SLICES_DIR / f'{patient_id}_slice{center_slice}_eat_neighborhood.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'saved to {save_path}')
    return save_path


def eat_neighborhood_gap(patient_id, center_slice, arms, eat_cls=EAT_CLS, window=EAT_NEIGHBOR_WINDOW):
    """Per arm: the center slice's EAT Dice vs. the mean EAT Dice of its valid
    neighbors (excluding the center; a neighbor with no EAT in GT is skipped,
    same convention as slice_scores()). This is the quantitative version of
    "can the model actually segment EAT nearby, just not here" -- a large
    positive gap (neighbors clearly better than the center) is the case for
    2.5D; a gap near zero means the whole neighborhood is uniformly hard,
    which argues against a missing-context explanation for that slice.
    Returns {label: (center_dice, neighbor_mean_dice, gap)}.
    """
    gaps = {}
    for label, run_name_fn in arms.items():
        pred_vol, gt_vol, _, _ = predict_patient(patient_id, run_name_fn)
        n_slices = pred_vol.shape[0]
        neighbor_dices = []
        for s in range(center_slice - window, center_slice + window + 1):
            if s == center_slice or not (0 <= s < n_slices) or not (gt_vol[s] == eat_cls).any():
                continue
            d = metrics.dice_binary(pred_vol[s] == eat_cls, gt_vol[s] == eat_cls)
            if np.isfinite(d):
                neighbor_dices.append(d)
        center_dice = metrics.dice_binary(pred_vol[center_slice] == eat_cls, gt_vol[center_slice] == eat_cls)
        neighbor_mean = float(np.mean(neighbor_dices)) if neighbor_dices else float('nan')
        gap = neighbor_mean - center_dice if neighbor_dices else float('nan')
        gaps[label] = (center_dice, neighbor_mean, gap)
    return gaps


worst_eat_slices = pool_class_slice_ranking(RUN_NAME_FN[NEIGHBORHOOD_ARM], EAT_CLS)[:N_EAT_NEIGHBORHOOD_SWEEP]
if not worst_eat_slices:
    print(f'No fold has finished for {NEIGHBORHOOD_ARM} yet.')
else:
    print(f'quantitative check across the worst {len(worst_eat_slices)} EAT slices '
          f'(ranked by {NEIGHBORHOOD_ARM}):\n')
    print(f'{"patient":<20}{"slice":>6}   {"arm":<18}{"center":>8}{"neighbors":>11}{"gap":>9}')
    all_gaps = []
    for score, pid, s, fold in worst_eat_slices:
        gaps = eat_neighborhood_gap(pid, s, ALL_ARMS)
        for i, (label, (center_d, neigh_mean, gap)) in enumerate(gaps.items()):
            lead = f'{pid:<20}{s:>6}' if i == 0 else f'{"":<20}{"":>6}'
            neigh_str = f'{neigh_mean:>11.3f}' if np.isfinite(neigh_mean) else f'{"n/a":>11}'
            gap_str = f'{gap:>+9.3f}' if np.isfinite(gap) else f'{"n/a":>9}'
            print(f'{lead}   {label:<18}{center_d:>8.3f}{neigh_str}{gap_str}')
            if np.isfinite(gap):
                all_gaps.append(gap)
        print()

    if all_gaps:
        big_gap = sum(g > 0.10 for g in all_gaps)
        print(f'mean gap across {len(all_gaps)} (slice, arm) pairs: {np.mean(all_gaps):+.3f}  '
              f'-- {big_gap}/{len(all_gaps)} pairs have neighbors >0.10 Dice better than the center.')
        print('Large, repeated positive gaps are the quantitative case for 2.5D: the model can '
              'clearly segment EAT one slice away, just not on the failing slice. Gaps clustered '
              'near zero mean these neighborhoods are uniformly hard, which points more toward a '
              'structure-level (contrast/boundary) limitation than a missing-context one.')

    for score, pid, s, fold in worst_eat_slices:
        show_eat_slice_neighborhood(pid, s, ALL_ARMS,
                                     title_extra=f'  [{NEIGHBORHOOD_ARM} EAT Dice={score:.3f}]')

## Final: score the winner on the test set, once

Everything above pools validation-stage patients only; the test set has not
been touched. Run this **after** the winning arm is decided from the pooled
comparison above, and run it once.

k-fold gives you `K_FOLDS` trained models per arm, not one -- there is no
single "the" model the way a fixed single split has. Ensembling the K models
is a legitimate alternative, but it changes what "test performance" measures
(an ensemble, not a single model) and is not what any other notebook in this
project does. To stay comparable with `loss_ablation.ipynb` and
`both_phase_ablation.ipynb`, this cell scores test with ONE fold's weights --
the fold with the best validation macro Dice among the winning arm -- the
same way those notebooks score test with one seed's `best.pth`.

In [ ]:
WINNING_ARM = BASELINE_ARM     # <- set to whichever arm won on the pooled comparison
RUN_FINAL_TEST = False         # <- flip to True deliberately, once

if RUN_FINAL_TEST:
    # pick the fold with the best validation macro Dice among the winning arm
    best_fold, best_score = None, -float('inf')
    for f in range(K_FOLDS):
        cfg_path = run_dir_for(RUN_NAME_FN[WINNING_ARM](f)) / 'run_config.json'
        cfg = json.loads(cfg_path.read_text())
        if cfg.get('best_val_macro_dice', -float('inf')) > best_score:
            best_fold, best_score = f, cfg['best_val_macro_dice']
    print(f'Scoring test with {WINNING_ARM} fold {best_fold} '
          f'(val macro Dice {best_score:.4f})')

    run_name = RUN_NAME_FN[WINNING_ARM](best_fold)
    run_dir = run_dir_for(run_name)
    _, _, test_idx = train.split_patients_kfold(
        dataset, fold=best_fold, k_folds=K_FOLDS, test_percent=TEST_PERCENT, seed=SPLIT_SEED
    )

    model = UNet(n_channels=N_CHANNELS, n_classes=N_CLASSES, bilinear=False)
    model = model.to(memory_format=torch.channels_last).to(device=device)

    state_dict = torch.load(run_dir / 'best.pth', map_location=device)
    state_dict.pop('mask_values', None)      # injected by train.py, not a real weight
    model.load_state_dict(state_dict)
    model.eval()

    rows, summary = metrics.report(
        model, dataset, test_idx, device,
        n_classes=N_CLASSES,
        out_dir=run_dir, split_name='test',
        class_names=train.PHASE_CLASS_NAMES[PHASE],
        batch_size=BATCH_SIZE,
    )
    print(metrics.format_summary(summary))
else:
    print('RUN_FINAL_TEST is False -- flip it deliberately once the winner is decided.')
